# CARL Research Evidence — Covariance Methodology

## Objective

This notebook converts the validated CARL covariance experiments into
compact research evidence tables.

The analysis evaluates three covariance estimators:

- Sample covariance
- Fixed-intensity shrinkage
- Ledoit-Wolf shrinkage

The evidence is based on strictly out-of-sample walk-forward
experiments.

The objective is not simply to identify the estimator with the highest
return, but to determine whether the preferred covariance methodology
depends on the research objective.

In [1]:
import pandas as pd
import numpy as np

from crypto_alpha_lab.data import load_prices

from crypto_alpha_lab.research.covariance_experiment import (
    run_covariance_experiment,
)

from crypto_alpha_lab.evaluation.covariance_comparison import (
    compare_covariance_methods,
)

In [2]:
assets = {
    "BTC": "BTC-USD",
    "ETH": "ETH-USD",
    "SOL": "SOL-USD",
}

close_prices = pd.concat(
    {
        name: load_prices(
            ticker,
            "2021-01-01",
            "2025-12-31",
            refresh=False,
        )["Close"]
        for name, ticker in assets.items()
    },
    axis=1,
).dropna()

close_prices.shape

(1825, 3)

In [3]:
common = {
    "prices": close_prices,
    "train_size": 252,
    "test_size": 21,
}

experiments = [
    run_covariance_experiment(
        method="sample",
        **common,
    ),

    run_covariance_experiment(
        method="shrinkage",
        shrinkage=0.25,
        **common,
    ),

    run_covariance_experiment(
        method="ledoit_wolf",
        **common,
    ),
]

In [4]:
comparison = compare_covariance_methods(
    experiments,
    periods_per_year=252,
    risk_free_rate=0.0,
)

In [5]:
decision_table = (
    comparison.summary[
        [
            "total_return",
            "annualized_return",
            "annualized_volatility",
            "sharpe_ratio",
            "sortino_ratio",
            "maximum_drawdown",
            "calmar_ratio",
            "average_turnover",
        ]
    ]
    .copy()
)

decision_table

,total_return,annualized_return,annualized_volatility,sharpe_ratio,sortino_ratio,maximum_drawdown,calmar_ratio,average_turnover
method,,,,,,,,
sample,0.869704,0.106805,0.436643,0.450538,0.041746,-0.781959,0.143885,1.219442
shrinkage,0.781683,0.098184,0.435377,0.432726,0.039903,-0.767388,0.136013,1.110687
ledoit_wolf,0.568794,0.075755,0.437010,0.385554,0.035355,-0.725831,0.113913,1.352342


In [6]:
leaders = pd.Series(
    {
        "total_return": decision_table["total_return"].idxmax(),
        "annualized_return": decision_table["annualized_return"].idxmax(),
        "sharpe_ratio": decision_table["sharpe_ratio"].idxmax(),
        "sortino_ratio": decision_table["sortino_ratio"].idxmax(),
        "maximum_drawdown": decision_table["maximum_drawdown"].idxmax(),
        "calmar_ratio": decision_table["calmar_ratio"].idxmax(),
        "average_turnover": decision_table["average_turnover"].idxmin(),
    },
    name="leader",
)

leaders

total_return              sample
annualized_return         sample
sharpe_ratio              sample
sortino_ratio             sample
maximum_drawdown     ledoit_wolf
calmar_ratio              sample
average_turnover       shrinkage
Name: leader, dtype: str

In [7]:
evidence_matrix = pd.DataFrame(
    {
        "Performance": [
            "Strongest",
            "Competitive",
            "Weakest",
        ],
        "Risk Control": [
            "Weakest",
            "Strong",
            "Strongest",
        ],
        "Implementation": [
            "Moderate",
            "Strongest",
            "Weakest",
        ],
    },
    index=[
        "sample",
        "shrinkage",
        "ledoit_wolf",
    ],
)

evidence_matrix

,Performance,Risk Control,Implementation
sample,Strongest,Weakest,Moderate
shrinkage,Competitive,Strong,Strongest
ledoit_wolf,Weakest,Strongest,Weakest


In [8]:
from crypto_alpha_lab.research.covariance_robustness import (
    CovarianceRobustnessAnalyzer,
    RobustnessConfiguration,
)

In [9]:
configurations = [
    RobustnessConfiguration(
        train_size=126,
        test_size=21,
        shrinkage=0.25,
    ),
    RobustnessConfiguration(
        train_size=252,
        test_size=21,
        shrinkage=0.25,
    ),
    RobustnessConfiguration(
        train_size=504,
        test_size=21,
        shrinkage=0.25,
    ),
]

In [10]:
close_prices.shape

(1825, 3)

In [11]:
robustness_analyzer = CovarianceRobustnessAnalyzer(
    close_prices
)

In [12]:
robustness = robustness_analyzer.run(
    configurations=configurations,
    methods=[
        "sample",
        "shrinkage",
        "ledoit_wolf",
    ],
)

In [13]:
robustness.summary

,method,train_size,test_size,shrinkage,total_return,observation_count,fold_count
0,sample,126,21,NaN,0.822511,1680,80
1,shrinkage,126,21,0.25,0.650715,1680,80
2,ledoit_wolf,126,21,NaN,0.191804,1680,80
3,sample,252,21,NaN,0.869704,1554,74
4,shrinkage,252,21,0.25,0.781683,1554,74
5,ledoit_wolf,252,21,NaN,0.568794,1554,74
6,sample,504,21,NaN,1.948960,1302,62
7,shrinkage,504,21,0.25,1.758280,1302,62
8,ledoit_wolf,504,21,NaN,1.276179,1302,62


In [14]:
robustness.metadata

{'analysis': 'covariance_robustness',
 'out_of_sample': True,
 'methods': ['sample', 'shrinkage', 'ledoit_wolf'],
 'configuration_count': 3,
 'experiment_count': 9}

In [15]:
{
    'analysis': 'covariance_robustness',
    'out_of_sample': True,
    'methods': ['sample', 'shrinkage', 'ledoit_wolf'],
    'configuration_count': 3,
    'experiment_count': 9
}

{'analysis': 'covariance_robustness',
 'out_of_sample': True,
 'methods': ['sample', 'shrinkage', 'ledoit_wolf'],
 'configuration_count': 3,
 'experiment_count': 9}

In [16]:
robustness_table = (
    robustness.summary[
        [
            "train_size",
            "method",
            "total_return",
            "fold_count",
            "observation_count",
        ]
    ]
    .copy()
)

robustness_table

,train_size,method,total_return,fold_count,observation_count
0,126,sample,0.822511,80,1680
1,126,shrinkage,0.650715,80,1680
2,126,ledoit_wolf,0.191804,80,1680
3,252,sample,0.869704,74,1554
4,252,shrinkage,0.781683,74,1554
5,252,ledoit_wolf,0.568794,74,1554
6,504,sample,1.948960,62,1302
7,504,shrinkage,1.758280,62,1302
8,504,ledoit_wolf,1.276179,62,1302


## Training-Window Robustness

To test whether the covariance-method ranking depends on a particular
amount of historical training data, the experiment was repeated using
126-, 252-, and 504-observation training windows.

The test window was held constant at 21 observations.

All experiments remain strictly out-of-sample: covariance matrices are
estimated using training data only, portfolio weights are constructed
from those estimates, and performance is evaluated on subsequent
unseen observations.

### Finding

Sample covariance produced the highest total return across all three
training-window configurations.

Shrinkage ranked second across all three configurations, while
Ledoit-Wolf ranked third.

However, the magnitude of the performance differences changes with
the training-window length. Therefore, the evidence supports a
persistent ranking in this experiment, but does not establish that
Sample covariance is universally superior.

In [17]:
ranking_table = (
    robustness_table
    .assign(
        rank=lambda df: (
            df.groupby("train_size")["total_return"]
            .rank(
                ascending=False,
                method="min",
            )
        )
    )
    .sort_values(
        ["train_size", "rank"]
    )
)

ranking_table

,train_size,method,total_return,fold_count,observation_count,rank
0,126,sample,0.822511,80,1680,1.0
1,126,shrinkage,0.650715,80,1680,2.0
2,126,ledoit_wolf,0.191804,80,1680,3.0
3,252,sample,0.869704,74,1554,1.0
4,252,shrinkage,0.781683,74,1554,2.0
5,252,ledoit_wolf,0.568794,74,1554,3.0
6,504,sample,1.948960,62,1302,1.0
7,504,shrinkage,1.758280,62,1302,2.0
8,504,ledoit_wolf,1.276179,62,1302,3.0


### Ranking Stability

The covariance-method ranking is unchanged across all three
training-window configurations:

1. Sample
2. Shrinkage
3. Ledoit-Wolf

This provides evidence that the baseline performance ordering is not
driven solely by the 252-observation training window.

In [18]:
ranking_stability = (
    ranking_table
    .groupby("method")["rank"]
    .agg(
        mean_rank="mean",
        worst_rank="max",
        best_rank="min",
    )
)

ranking_stability

,mean_rank,worst_rank,best_rank
method,,,
ledoit_wolf,3.0,3.0,3.0
sample,1.0,1.0,1.0
shrinkage,2.0,2.0,2.0


## Interpretation

The robustness analysis strengthens the baseline comparison.

Sample covariance retains first position across every training-window
configuration tested. Shrinkage consistently occupies second position,
while Ledoit-Wolf remains third.

The stability of this ranking suggests that the baseline result is not
specific to one arbitrary training-window choice.

Nevertheless, the results remain conditional on the CARL experimental
universe, portfolio construction method, asset universe, evaluation
period, and tested training-window configurations.

The robustness analysis therefore increases confidence in the observed
result without converting it into a universal claim about covariance
estimation.

In [19]:
return_pivot = (
    robustness_table
    .pivot(
        index="train_size",
        columns="method",
        values="total_return",
    )
)

return_pivot["sample_minus_shrinkage"] = (
    return_pivot["sample"]
    - return_pivot["shrinkage"]
)

return_pivot["sample_minus_ledoit_wolf"] = (
    return_pivot["sample"]
    - return_pivot["ledoit_wolf"]
)

return_pivot

method,ledoit_wolf,sample,shrinkage,sample_minus_shrinkage,sample_minus_ledoit_wolf
train_size,,,,,
126,0.191804,0.822511,0.650715,0.171796,0.630707
252,0.568794,0.869704,0.781683,0.088021,0.300910
504,1.276179,1.948960,1.758280,0.190680,0.672781


In [20]:
baseline_evidence = pd.DataFrame(
    {
        "baseline_return": decision_table["total_return"],
        "annualized_return": decision_table["annualized_return"],
        "sharpe_ratio": decision_table["sharpe_ratio"],
        "maximum_drawdown": decision_table["maximum_drawdown"],
        "average_turnover": decision_table["average_turnover"],
    }
)

baseline_evidence

,baseline_return,annualized_return,sharpe_ratio,maximum_drawdown,average_turnover
method,,,,,
sample,0.869704,0.106805,0.450538,-0.781959,1.219442
shrinkage,0.781683,0.098184,0.432726,-0.767388,1.110687
ledoit_wolf,0.568794,0.075755,0.385554,-0.725831,1.352342


In [21]:
robustness_ranking = (
    ranking_table
    .groupby("method")["rank"]
    .agg(
        mean_rank="mean",
        best_rank="min",
        worst_rank="max",
    )
)

robustness_ranking

,mean_rank,best_rank,worst_rank
method,,,
ledoit_wolf,3.0,3.0,3.0
sample,1.0,1.0,1.0
shrinkage,2.0,2.0,2.0


In [22]:
consolidated_evidence = baseline_evidence.join(
    robustness_ranking
)

consolidated_evidence

,baseline_return,annualized_return,sharpe_ratio,maximum_drawdown,average_turnover,mean_rank,best_rank,worst_rank
method,,,,,,,,
sample,0.869704,0.106805,0.450538,-0.781959,1.219442,1.0,1.0,1.0
shrinkage,0.781683,0.098184,0.432726,-0.767388,1.110687,2.0,2.0,2.0
ledoit_wolf,0.568794,0.075755,0.385554,-0.725831,1.352342,3.0,3.0,3.0


In [23]:
scenario_scores = pd.DataFrame(
    {
        "performance_focused": {
            "sample": 0.754996,
            "shrinkage": 0.654528,
            "ledoit_wolf": 0.200000,
        },
        "risk_control_focused": {
            "sample": 0.454996,
            "shrinkage": 0.517402,
            "ledoit_wolf": 0.500000,
        },
        "implementation_focused": {
            "sample": 0.624979,
            "shrinkage": 0.790245,
            "ledoit_wolf": 0.150000,
        },
    }
)

scenario_scores

,performance_focused,risk_control_focused,implementation_focused
sample,0.754996,0.454996,0.624979
shrinkage,0.654528,0.517402,0.790245
ledoit_wolf,0.200000,0.500000,0.150000


In [24]:
scenario_winner = scenario_scores.idxmax(
    axis=0
)

scenario_winner

performance_focused          sample
risk_control_focused      shrinkage
implementation_focused    shrinkage
dtype: str

In [25]:
scenario_scores = pd.DataFrame(
    scenario_scores
)

scenario_scores

,performance_focused,risk_control_focused,implementation_focused
sample,0.754996,0.454996,0.624979
shrinkage,0.654528,0.517402,0.790245
ledoit_wolf,0.200000,0.500000,0.150000


In [26]:
scenario_winner = scenario_scores.idxmax(axis=0)

scenario_winner

performance_focused          sample
risk_control_focused      shrinkage
implementation_focused    shrinkage
dtype: str

In [27]:
print("Scenario winners:")
for scenario, method in scenario_winner.items():
    print(f"{scenario}: {method}")

Scenario winners:
performance_focused: sample
risk_control_focused: shrinkage
implementation_focused: shrinkage


## — Final Decision Matrix

The purpose of this section is to convert the empirical evidence from
the covariance comparison, scenario analysis, and robustness analysis
into an explicit research decision.

Rather than selecting a universally "best" covariance estimator, we
evaluate each methodology according to three practical objectives:

1. **Performance-focused** — prioritize return and risk-adjusted performance.
2. **Risk-control-focused** — prioritize drawdown and stability.
3. **Implementation-focused** — prioritize turnover and practical portfolio implementation.

This framework recognizes that covariance estimation is ultimately a
portfolio-construction decision, and the preferred estimator may depend
on the objective of the portfolio.

In [28]:
decision_matrix = pd.DataFrame(
    {
        "objective": [
            "Performance-focused",
            "Risk-control-focused",
            "Implementation-focused",
        ],
        "preferred_method": [
            scenario_winner["performance_focused"],
            scenario_winner["risk_control_focused"],
            scenario_winner["implementation_focused"],
        ],
        "decision_basis": [
            "Highest performance-oriented scenario score",
            "Highest risk-control-oriented scenario score",
            "Highest implementation-oriented scenario score",
        ],
    }
)

decision_matrix

,objective,preferred_method,decision_basis
0,Performance-focused,sample,Highest performance-oriented scenario score
1,Risk-control-focused,shrinkage,Highest risk-control-oriented scenario score
2,Implementation-focused,shrinkage,Highest implementation-oriented scenario score




### Interpretation

The decision matrix produces a differentiated conclusion.

**Sample covariance** is preferred when the primary objective is
performance. It achieved the strongest performance-focused scenario
score and also produced the highest total and annualized returns and
Sharpe ratio in the baseline comparison.

**Shrinkage covariance** is preferred when risk control is emphasized.
Although it does not maximize return, it provides a more conservative
risk profile and achieves the strongest risk-control scenario score.

Shrinkage is also preferred for implementation. Its lower average
turnover relative to the other methods makes it more attractive when
portfolio implementation and trading activity are important.

Therefore, the research does not support the claim that one covariance
estimator dominates under every objective.

In [29]:
decision_evidence = (
    comparison.summary[
        [
            "total_return",
            "annualized_return",
            "annualized_volatility",
            "sharpe_ratio",
            "maximum_drawdown",
            "calmar_ratio",
            "average_turnover",
        ]
    ]
    .copy()
)

decision_evidence

,total_return,annualized_return,annualized_volatility,sharpe_ratio,maximum_drawdown,calmar_ratio,average_turnover
method,,,,,,,
sample,0.869704,0.106805,0.436643,0.450538,-0.781959,0.143885,1.219442
shrinkage,0.781683,0.098184,0.435377,0.432726,-0.767388,0.136013,1.110687
ledoit_wolf,0.568794,0.075755,0.437010,0.385554,-0.725831,0.113913,1.352342


### Baseline Evidence

The baseline out-of-sample comparison provides the following key
evidence:

- Sample covariance generated the highest total return.
- Sample covariance generated the highest annualized return.
- Sample covariance generated the highest Sharpe ratio.
- Ledoit-Wolf produced the least severe maximum drawdown.
- Shrinkage produced the lowest average turnover.
- The differences therefore represent trade-offs rather than universal dominance.

These findings are particularly important because the comparison is
based on strictly out-of-sample walk-forward results rather than
in-sample portfolio optimization.

robustness_evidence = (
    robustness.summary[
        [
            "method",
            "train_size",
            "test_size",
            "total_return",
            "fold_count",
        ]
    ]
    .copy()
)

robustness_evidence

### Robustness Evidence

The robustness analysis evaluates whether the conclusions remain stable
when the training-window length changes.

Three training windows were examined:

- 126 observations
- 252 observations
- 504 observations

Each configuration was evaluated using the same 21-observation test
window and the same three covariance methodologies.

The analysis therefore contains nine strictly out-of-sample experiments.

A consistent pattern emerges:

- Sample covariance produced the highest total return at all three
  training-window lengths.
- Shrinkage remained below sample covariance in return but above
  Ledoit-Wolf.
- Ledoit-Wolf produced the lowest total return across all three
  training-window configurations.

This strengthens the evidence that the performance ranking observed
in the baseline experiment is not solely attributable to one particular
training-window choice.

In [30]:
final_decision = {
    "performance": scenario_winner["performance_focused"],
    "risk_control": scenario_winner["risk_control_focused"],
    "implementation": scenario_winner["implementation_focused"],
}

final_decision

{'performance': 'sample',
 'risk_control': 'shrinkage',
 'implementation': 'shrinkage'}

In [31]:
# Expected 

{
    "performance": "sample",
    "risk_control": "shrinkage",
    "implementation": "shrinkage",
}

{'performance': 'sample',
 'risk_control': 'shrinkage',
 'implementation': 'shrinkage'}

## Final Research Conclusion

The evidence does not support selecting a single covariance estimator
as universally optimal.

Instead, the preferred methodology depends on the portfolio objective.

For a **performance-oriented portfolio**, the sample covariance estimator
is preferred because it produced the strongest observed return and
risk-adjusted performance across the baseline and robustness analyses.

For a **risk-control-oriented portfolio**, shrinkage covariance is
preferred because it provides a more conservative risk profile while
remaining competitive on performance.

For an **implementation-oriented portfolio**, shrinkage is preferred
because it produced lower average turnover than the alternative
methods, reducing the intensity of portfolio rebalancing.

The central research finding is therefore:

> **Covariance estimation involves a trade-off between performance,
> risk control, and implementation efficiency rather than a universally
> dominant estimator.**

This conclusion is based on strictly out-of-sample walk-forward
experiments and robustness tests across multiple training-window
configurations.

------------------------

## Research Evidence Summary

This section summarizes the principal empirical evidence generated by
the covariance research.

The analysis combines:

- baseline out-of-sample performance,
- scenario-based decision analysis,
- training-window robustness,
- and implementation considerations.

The purpose is not to introduce another experiment, but to provide a
compact evidence trail supporting the final research conclusion.

In [32]:
import pandas as pd
evidence_summary = pd.DataFrame(
    {
        "baseline_performance_winner": [
            scenario_winner["performance_focused"]
        ],
        "risk_control_winner": [
            scenario_winner["risk_control_focused"]
        ],
        "implementation_winner": [
            scenario_winner["implementation_focused"]
        ],
        "robustness_configurations": [
            robustness.metadata["configuration_count"]
        ],
        "robustness_experiments": [
            robustness.metadata["experiment_count"]
        ],
        "out_of_sample": [
            robustness.metadata["out_of_sample"]
        ],
    }
)

evidence_summary

,baseline_performance_winner,risk_control_winner,implementation_winner,robustness_configurations,robustness_experiments,out_of_sample
0,sample,shrinkage,shrinkage,3,9,True


### Evidence Interpretation

The evidence supports a conditional rather than universal selection
rule.

Sample covariance is the preferred methodology when maximizing observed
portfolio performance is the primary objective.

Shrinkage becomes preferable when the research objective places greater
emphasis on risk control or implementation efficiency.

The robustness analysis also shows that the covariance comparison is not
restricted to a single training-window configuration. Nine out-of-sample
experiments were evaluated across three training-window lengths and
three covariance methodologies.

Consequently, the final recommendation is objective-dependent rather
than based on a single performance metric.

## Research Limitations

Several limitations should be considered when interpreting the results.

1. **Limited asset universe**

   The baseline experiment uses a small cryptocurrency universe
   consisting of BTC, ETH, and SOL. Results may therefore not generalize
   to a broader cross-section of crypto assets.

2. **Historical dependence**

   The conclusions are based on the selected historical sample and may
   change under different market regimes.

3. **Portfolio objective**

   The experiments use a global minimum-variance portfolio construction.
   Different portfolio objectives or constraints could produce different
   rankings.

4. **Transaction-cost assumptions**

   Implementation conclusions depend on the transaction-cost framework
   used by the experiment.

5. **Covariance methodology**

   Only three covariance estimators are evaluated:
   sample covariance, explicit shrinkage, and Ledoit-Wolf shrinkage.

6. **Model risk**

   Superior historical out-of-sample performance does not guarantee
   superior future performance.

These limitations do not invalidate the experiment. Instead, they define
the scope within which its conclusions should be interpreted.

-------------------------------

## Final Research Statement

### Research Question

Which covariance-estimation methodology provides the most useful
portfolio construction characteristics under different investment
objectives?

### Evidence

Three covariance methodologies were evaluated using strictly
out-of-sample walk-forward experiments:

- Sample covariance
- Fixed-intensity shrinkage
- Ledoit-Wolf shrinkage

The baseline comparison evaluated return, volatility, Sharpe ratio,
Sortino ratio, drawdown, Calmar ratio, hit rate, and turnover.

The robustness analysis subsequently varied the training-window length
across 126, 252, and 504 observations while preserving the 21-observation
test window.

### Finding

No covariance methodology dominates across every objective.

Sample covariance is preferred for performance-focused objectives,
while shrinkage is preferred for risk-control and implementation-focused
objectives.

### Research Implication

The appropriate covariance estimator should therefore be selected as a
function of the portfolio's objective and implementation constraints,
rather than assuming that one estimator is universally superior.

### Confidence

The conclusion is supported by multiple strictly out-of-sample
experiments and sensitivity analysis, but remains conditional on the
asset universe, historical period, portfolio construction framework,
and transaction-cost assumptions used in this study.